# AnimalCLEF open-set roles

Use the canonical image catalogues from 00_prepare_benchmark.ipynb. Assign whole observations to gallery, known query, or novel query with the seed-42 protocol. Private jaguar data use only novel queries. Similarity computation follows in 03a_wildfusion.ipynb.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("APEXID_REPO_ROOT", Path.cwd())).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "apexid").is_dir():
    raise FileNotFoundError("Start the notebook in the repository or set APEXID_REPO_ROOT.")
sys.path.insert(0, str(REPO_ROOT))

import hashlib
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

In [ ]:
# ============================================================
# USER KNOBS
# ============================================================
ALLOWED_DATASETS = (
    "bristol_gorillas",
    "elephants",
    "gcn_id",
    "jaguar_antonio",
)
# "all" writes every dataset. Set to one name to rebuild a single CSV.
DATASET_NAME = "all"
SPLIT_SEED = 42
UNKNOWN_IDENTITY_FRAC = 0.20
GALLERY_GROUP_FRAC = 0.80
DINO_THRESHOLD = 0.98
DINO_BATCH_SIZE = 8
JAGUAR_NAME = "jaguar_antonio"

if DATASET_NAME != "all" and DATASET_NAME not in ALLOWED_DATASETS:
    raise ValueError(f"DATASET_NAME must be 'all' or one of {ALLOWED_DATASETS}")

STAGE00_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_PREPARED_INPUT", "")),
    REPO_ROOT / "outputs" / "apexid_cvpr_prepared",
)
OUTPUT_ROOT = Path(os.environ.get(
    "APEXID_STAGE03_SPLITS_OUTPUT",
    REPO_ROOT / "outputs" / "stage03" / "03_splits",
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def first_root(candidates, marker):
    for candidate in candidates:
        if str(candidate) and (candidate / marker).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {marker}; checked: {[str(x) for x in candidates]}")


STAGE00_ROOT = first_root(STAGE00_ROOT_CANDIDATES, "preparation_manifest.json")
SELECTED_DATASETS = ALLOWED_DATASETS if DATASET_NAME == "all" else (DATASET_NAME,)
print("SELECTED_DATASETS:", SELECTED_DATASETS)
print("SPLIT_SEED:", SPLIT_SEED)
print("UNKNOWN_IDENTITY_FRAC:", UNKNOWN_IDENTITY_FRAC)
print("GALLERY_GROUP_FRAC:", GALLERY_GROUP_FRAC)
print("STAGE00_ROOT:", STAGE00_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# Load Stage 00 canonical manifests and resolve live image paths for grouping.
import importlib.util

from apexid.data.split_v2 import (
    FeatureSpec,
    UnionFind,
    _union_equal,
    build_features_by_index,
    extract_feature_cache,
    hard_observation_components,
    parse_strict_date,
    sha256_file,
    stable_id,
)

from apexid.data.encounters import (
    _normalise_elephant_date,
    infer_bristol_encounter,
    infer_elephant_encounter,
)
from apexid.evaluation.embeddings import read_path_map, remap_path

PREPARATION_MANIFEST = json.loads((STAGE00_ROOT / "preparation_manifest.json").read_text(encoding="utf-8"))
if PREPARATION_MANIFEST.get("status") != "passed":
    raise RuntimeError("Stage 00 preparation_manifest.json is not marked passed")
METADATA_SHA256 = sha256_file(STAGE00_ROOT / "preparation_manifest.json")

PATH_MAP = read_path_map(os.environ.get("APEXID_PATH_MAP"))
IMAGE_ROOT = Path(os.environ.get("APEXID_IMAGE_ROOT", REPO_ROOT))

def resolve_image_path(frozen_path: str) -> Path:
    path = Path(remap_path(frozen_path, PATH_MAP))
    path = path if path.is_absolute() else IMAGE_ROOT / path
    if not path.is_file():
        raise FileNotFoundError(f"Mount {frozen_path} or set APEXID_PATH_MAP.")
    return path.resolve()

def load_canonical(dataset_name: str) -> pd.DataFrame:
    path = STAGE00_ROOT / "manifests" / f"{dataset_name}_canonical_manifest.csv"
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, dtype=str).fillna("")
    required = {"image_id", "path", "identity", "dataset"}
    if not required <= set(frame.columns):
        raise ValueError(f"{dataset_name}: missing columns {sorted(required - set(frame.columns))}")
    if frame["image_id"].duplicated().any():
        raise ValueError(f"{dataset_name}: duplicate image_id")
    if set(frame["dataset"].astype(str)) - {dataset_name, ""}:
        raise ValueError(f"{dataset_name}: canonical dataset column is mixed")
    frame["dataset"] = dataset_name
    return frame.reset_index(drop=True)


frames = {name: load_canonical(name) for name in SELECTED_DATASETS}
for name, frame in frames.items():
    print(name, "images:", len(frame), "identities:", frame["identity"].nunique())

In [ ]:
# Split V2 grouping: time-aware then explicit encounters then DINOv2, then 80/20 roles.
import re
from datetime import date as dt_date

FEATURE = FeatureSpec()
CACHE_ROOT = OUTPUT_ROOT / "_dino_cache"
SPLIT_COLUMNS = [
    "image_id", "path", "identity", "dataset", "observation_group_id",
    "open_role", "grouping_method", "identity_status", "split_seed",
]
ISO_YMD = re.compile(r"(?<!\d)((?:19|20)\d{2})[-_.](\d{2})[-_.](\d{2})(?!\d)")
YMD_COMPACT = re.compile(r"(?<!\d)((?:19|20)\d{2})(\d{2})(\d{2})(?!\d)")
DMY = re.compile(r"(?<!\d)(\d{2})[-_.](\d{2})[-_.]((?:19|20)\d{2})(?!\d)")
FRAME_SUFFIX = re.compile(
    r"(?:(?:[_-](?:img|image|frame|photo|pic|f)?[_-]?\d+)|(?:\(\d+\))|(?:_(?:left|right)(?:photo)?))+$",
    re.I,
)


def _valid_ymd(year, month, day) -> str:
    try:
        return dt_date(int(year), int(month), int(day)).isoformat()
    except ValueError:
        return ""


def parse_name_date(path: str, metadata_date: str = "") -> str:
    parsed, _label = parse_strict_date(metadata_date)
    if parsed:
        return parsed
    parts = [Path(path).name, Path(path).stem, *Path(path).parts[-4:]]
    text = " ".join(str(part) for part in parts)
    for match in ISO_YMD.finditer(text):
        iso = _valid_ymd(*match.groups())
        if iso:
            return iso
    for match in DMY.finditer(text):
        day, month, year = match.groups()
        iso = _valid_ymd(year, month, day)
        if iso:
            return iso
    for match in YMD_COMPACT.finditer(text):
        iso = _valid_ymd(*match.groups())
        if iso:
            return iso
    token = _normalise_elephant_date(Path(path).stem)
    parsed, _label = parse_strict_date(token or "")
    return parsed


def dataset_name_token(dataset: str, identity: str, path: str) -> str:
    filename = Path(path).name
    if dataset == "bristol_gorillas":
        key, ok = infer_bristol_encounter(identity, filename)
        if ok:
            return key
    elif dataset == "elephants":
        key, ok = infer_elephant_encounter(identity, filename)
        if ok:
            return key
    stem = Path(filename).stem
    stripped = re.sub(rf"^{re.escape(identity)}[_-]+", "", stem, flags=re.I)
    core = FRAME_SUFFIX.sub("", stripped).strip("_- .")
    skip_parents = {identity.lower(), dataset.lower(), "images", "image", "data"}
    parent = Path(path).parent.name
    if (not core or core.lower() == identity.lower() or len(core) < 3) and parent.lower() not in skip_parents:
        core = FRAME_SUFFIX.sub("", parent).strip("_- .")
    generic_cores = {"img", "image", "images", "dsc", "photo", "pic", "p", "f", "frame", "im", "scan"}
    if core.lower() in generic_cores:
        core = ""
    if not core or core.lower() == identity.lower() or len(core) < 3:
        return ""
    return f"{dataset}:{identity}:name:{core.lower()}"


def cvpr_explicit_source(dataset: str, identity: str, path: str) -> str:
    return dataset_name_token(dataset, identity, path)


def cvpr_date(dataset: str, row: dict) -> str:
    del dataset
    return parse_name_date(str(row.get("path", "")), str(row.get("date", "") or ""))


def stable_sorted(values, *parts):
    return sorted(values, key=lambda value: stable_id("ord", SPLIT_SEED, *parts, str(value)))


def rows_from_frame(frame: pd.DataFrame, dataset_name: str) -> list[dict]:
    rows = []
    for source_index, record in enumerate(frame.to_dict(orient="records")):
        path = str(record["path"])
        identity = str(record["identity"])
        parsed_date = cvpr_date(dataset_name, record)
        explicit = cvpr_explicit_source(dataset_name, identity, path)
        digest = str(record.get("image_sha256", "") or "")
        rows.append({
            "image_id": str(record["image_id"]),
            "path": path,
            "identity": identity,
            "dataset": dataset_name,
            "date": str(record.get("date", "") or ""),
            "_idx": source_index,
            "_key": (dataset_name, identity),
            "_parsed_date": parsed_date,
            "_explicit_source": explicit,
            "_image_sha256": digest,
        })
    return rows


def ensure_hashes(rows: list[dict], dataset_name: str) -> None:
    missing = [row for row in rows if not row["_image_sha256"]]
    if not missing:
        return
    print(f"{dataset_name}: hashing {len(missing)} images without image_sha256")
    for row in missing:
        live = resolve_image_path(row["path"])
        row["_image_sha256"] = sha256_file(live)


def plan_method(items: list[dict]) -> str:
    dates = [str(row["_parsed_date"]) for row in items]
    explicit = [str(row["_explicit_source"]) for row in items]
    if all(dates) and len(set(dates)) >= 2:
        return "temporal"
    if all(explicit) and len(set(explicit)) >= 2:
        return "explicit_source"
    return "similarity_fallback"


def observation_groups(items: list[dict], method: str, features_by_index: dict) -> list[list[int]]:
    if method in {"temporal", "explicit_source"}:
        return hard_observation_components(items)
    indices = [int(row["_idx"]) for row in items]
    uf = UnionFind(indices)
    _union_equal(uf, items, "_image_sha256")
    _union_equal(uf, items, "_explicit_source")
    _union_equal(uf, items, "_parsed_date")
    if all(index in features_by_index for index in indices) and len(indices) >= 2:
        order = list(indices)
        matrix = np.asarray([features_by_index[index] for index in order], dtype=np.float32)
        similarity = matrix @ matrix.T
        for left in range(len(order)):
            hits = np.flatnonzero(similarity[left, left + 1:] > DINO_THRESHOLD) + left + 1
            for right in hits:
                uf.union(order[left], order[int(right)])
    return sorted(uf.groups(), key=lambda group: group[0])


def assign_known_roles(items: list[dict], groups: list[list[int]], method: str) -> None:
    index_to_row = {int(row["_idx"]): row for row in items}
    labeled = []
    for group in groups:
        group_rows = [index_to_row[index] for index in group]
        observation = stable_id(
            "obs",
            items[0]["dataset"],
            items[0]["identity"],
            method,
            sorted(row["image_id"] for row in group_rows),
        )
        min_date = min((row["_parsed_date"] or "9999-99-99") for row in group_rows)
        labeled.append((min_date, observation, group_rows))
    if method == "temporal":
        labeled.sort(key=lambda item: (item[0], item[1]))
    else:
        labeled.sort(key=lambda item: stable_id("grp", SPLIT_SEED, item[1]))
    n_gallery = min(len(labeled) - 1, max(1, int(round(len(labeled) * GALLERY_GROUP_FRAC))))
    for rank, (_date, observation, group_rows) in enumerate(labeled):
        role = "gallery" if rank < n_gallery else "query_known"
        for row in group_rows:
            row["observation_group_id"] = observation
            row["open_role"] = role
            row["grouping_method"] = method
            row["identity_status"] = "known"


def assign_uniform_role(items: list[dict], groups: list[list[int]], method: str, role: str, status: str) -> None:
    index_to_row = {int(row["_idx"]): row for row in items}
    for group in groups:
        group_rows = [index_to_row[index] for index in group]
        observation = stable_id(
            "obs",
            items[0]["dataset"],
            items[0]["identity"],
            method,
            sorted(row["image_id"] for row in group_rows),
        )
        for row in group_rows:
            row["observation_group_id"] = observation
            row["open_role"] = role
            row["grouping_method"] = method
            row["identity_status"] = status


def split_dataset(dataset_name: str, frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    rows = rows_from_frame(frame, dataset_name)
    if dataset_name == JAGUAR_NAME:
        for row in rows:
            row["observation_group_id"] = stable_id("obs", dataset_name, row["identity"], "query_only", row["image_id"])
            row["open_role"] = "query_novel"
            row["grouping_method"] = "query_only_full_pool"
            row["identity_status"] = "unknown"
            row["split_seed"] = str(SPLIT_SEED)
        out = pd.DataFrame(rows)
        return out, out.iloc[0:0].copy(), {
            "dataset": dataset_name,
            "protocol": "query_only_defaults",
            "images": len(out),
            "identities": int(out["identity"].nunique()),
            "known_identities": 0,
            "unknown_identities": int(out["identity"].nunique()),
            "excluded_identities": 0,
            "gallery_images": 0,
            "query_known_images": 0,
            "query_novel_images": len(out),
            "dino_identities": 0,
            "singleton_query_novel_identities": 0,
        }

    ensure_hashes(rows, dataset_name)
    by_identity = defaultdict(list)
    for row in rows:
        by_identity[row["identity"]].append(row)

    methods = {identity: plan_method(items) for identity, items in by_identity.items()}
    dino_rows = []
    for identity, items in by_identity.items():
        if methods[identity] != "similarity_fallback":
            continue
        if len(hard_observation_components(items)) >= 2:
            dino_rows.extend(items)

    features_by_index = {}
    if dino_rows:
        print(f"{dataset_name}: DINOv2 fallback on {len(dino_rows)} images / {len({row['identity'] for row in dino_rows})} identities")
        original_paths = []
        ordered_dino = sorted(dino_rows, key=lambda row: int(row["_idx"]))
        for row in ordered_dino:
            original_paths.append(row["path"])
            row["path"] = str(resolve_image_path(row["path"]))
        cache_dir = CACHE_ROOT / dataset_name
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        features = extract_feature_cache(
            cache_dir=cache_dir,
            candidate_rows=ordered_dino,
            metadata_sha256=METADATA_SHA256,
            feature=FEATURE,
            data_root=Path("/"),
            checkpoint=None,
            device=device,
            batch_size=DINO_BATCH_SIZE,
        )
        features_by_index = build_features_by_index(ordered_dino, np.asarray(features))
        for row, original in zip(ordered_dino, original_paths):
            row["path"] = original

    grouped = {}
    eligible = []
    singleton_ids = []
    for identity, items in by_identity.items():
        method = methods[identity]
        groups = observation_groups(items, method, features_by_index)
        grouped[identity] = (method, groups, items)
        if len(groups) >= 2:
            eligible.append(identity)
        else:
            singleton_ids.append(identity)

    kept = []
    if len(eligible) >= 2:
        protocol = "gallery_attach_optuna"
        n_unknown = min(len(eligible) - 1, max(1, int(round(len(eligible) * UNKNOWN_IDENTITY_FRAC))))
        unknown = set(stable_sorted(eligible, dataset_name)[:n_unknown])
        known = [identity for identity in eligible if identity not in unknown]
        for identity in known:
            method, groups, items = grouped[identity]
            assign_known_roles(items, groups, method)
            kept.extend(items)
        for identity in unknown:
            method, groups, items = grouped[identity]
            assign_uniform_role(items, groups, method, "query_novel", "unknown")
            kept.extend(items)
    elif len(eligible) == 1 and singleton_ids:
        protocol = "gallery_attach_optuna"
        identity = eligible[0]
        method, groups, items = grouped[identity]
        assign_known_roles(items, groups, method)
        kept.extend(items)
    else:
        protocol = "query_only_defaults"
        print(f"{dataset_name}: no leak-safe gallery (multi-group identities={len(eligible)}); all images are query_novel")
        for identity, (method, groups, items) in grouped.items():
            assign_uniform_role(items, groups, method, "query_novel", "unknown")
            kept.extend(items)

    if protocol == "gallery_attach_optuna":
        for identity in singleton_ids:
            method, groups, items = grouped[identity]
            assign_uniform_role(items, groups, method, "query_novel", "unknown")
            kept.extend(items)

    for row in kept:
        row["split_seed"] = str(SPLIT_SEED)

    out = pd.DataFrame(kept).sort_values("image_id", kind="stable").reset_index(drop=True)
    excluded = out.iloc[0:0].copy()
    singleton_novel = int(len(set(singleton_ids)))
    stats = {
        "dataset": dataset_name,
        "protocol": protocol,
        "images": len(out),
        "identities": int(out["identity"].nunique()),
        "known_identities": int(out.loc[out["identity_status"].eq("known"), "identity"].nunique()),
        "unknown_identities": int(out.loc[out["identity_status"].eq("unknown"), "identity"].nunique()),
        "excluded_identities": 0,
        "singleton_query_novel_identities": singleton_novel,
        "gallery_images": int(out["open_role"].eq("gallery").sum()),
        "query_known_images": int(out["open_role"].eq("query_known").sum()),
        "query_novel_images": int(out["open_role"].eq("query_novel").sum()),
        "dino_identities": int(sum(method == "similarity_fallback" for method, _groups, _items in grouped.values() if len(_groups) >= 2)),
    }
    return out, excluded, stats


summaries = []
for dataset_name in SELECTED_DATASETS:
    print(f"\n=== {dataset_name} ===")
    split_frame, excluded_frame, stats = split_dataset(dataset_name, frames[dataset_name])
    dataset_root = OUTPUT_ROOT / dataset_name
    dataset_root.mkdir(parents=True, exist_ok=True)
    public = split_frame[[column for column in SPLIT_COLUMNS if column in split_frame.columns]].copy()
    split_path = dataset_root / "open_role_split.csv"
    public.to_csv(split_path, index=False)
    for role in ("gallery", "query_known", "query_novel"):
        public.loc[public["open_role"].eq(role)].to_csv(dataset_root / f"{role}.csv", index=False)
    if len(excluded_frame):
        excluded_frame[[column for column in SPLIT_COLUMNS if column in excluded_frame.columns]].to_csv(
            dataset_root / "excluded_singleton_identities.csv", index=False
        )
    stats["split_csv"] = str(split_path)
    stats["split_sha256"] = hashlib.sha256(split_path.read_bytes()).hexdigest()
    summaries.append(stats)
    print(json.dumps(stats, indent=2, sort_keys=True))

summary_frame = pd.DataFrame(summaries)
summary_frame.to_csv(OUTPUT_ROOT / "split_summary.csv", index=False)
print("\nWrote", OUTPUT_ROOT)
print(summary_frame.to_string(index=False))

In [ ]:
# Leak-safe checks, then the hand-off manifest for 03a.
def validate_split(dataset_name: str, frame: pd.DataFrame, protocol: str) -> None:
    required = {"image_id", "path", "identity", "dataset", "open_role"}
    if not required <= set(frame.columns):
        raise ValueError(f"{dataset_name}: split CSV missing {sorted(required - set(frame.columns))}")
    if frame["image_id"].duplicated().any():
        raise ValueError(f"{dataset_name}: duplicate image_id")
    if set(frame["dataset"].astype(str)) != {dataset_name}:
        raise ValueError(f"{dataset_name}: mixed dataset column")
    roles = set(frame["open_role"].astype(str))
    if protocol == "query_only_defaults":
        if roles != {"query_novel"}:
            raise RuntimeError(f"{dataset_name}: gallery-free split must be 100% query_novel, got {roles}")
        return
    if not {"gallery", "query_known", "query_novel"} <= roles:
        raise RuntimeError(f"{dataset_name}: missing open_role values {roles}")
    gallery = frame.loc[frame["open_role"].eq("gallery")]
    query_known = frame.loc[frame["open_role"].eq("query_known")]
    query_novel = frame.loc[frame["open_role"].eq("query_novel")]
    if set(gallery["image_id"]) & set(frame.loc[frame["open_role"].ne("gallery"), "image_id"]):
        raise RuntimeError(f"{dataset_name}: gallery/query image overlap")
    known_ids = set(gallery["identity"]) | set(query_known["identity"])
    novel_ids = set(query_novel["identity"])
    if known_ids & novel_ids:
        raise RuntimeError(f"{dataset_name}: unknown identity leaked into gallery or query_known")
    if set(gallery["identity"]) != set(query_known["identity"]):
        raise RuntimeError(f"{dataset_name}: every known identity must have both gallery and query_known images")
    if not len(gallery) or not len(query_known) or not len(query_novel):
        raise RuntimeError(f"{dataset_name}: empty gallery or query partition")


manifest_datasets = {}
for stats in summaries:
    dataset_name = stats["dataset"]
    split_path = OUTPUT_ROOT / dataset_name / "open_role_split.csv"
    frame = pd.read_csv(split_path, dtype=str).fillna("")
    protocol = stats["protocol"]
    validate_split(dataset_name, frame, protocol)
    manifest_datasets[dataset_name] = {
        **stats,
        "open_role_counts": {str(role): int(count) for role, count in frame.groupby("open_role").size().items()},
        "grouping_method_counts": {
            str(method): int(count) for method, count in frame.groupby("grouping_method").size().items()
        },
    }

manifest = {
    "stage": "03_open_set_splits",
    "status": "passed",
    "split_seed": SPLIT_SEED,
    "unknown_identity_frac": UNKNOWN_IDENTITY_FRAC,
    "gallery_group_frac": GALLERY_GROUP_FRAC,
    "dino_model": FEATURE.model,
    "dino_revision": FEATURE.revision,
    "dino_threshold": DINO_THRESHOLD,
    "stage00_root": str(STAGE00_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "datasets": manifest_datasets,
}
manifest_path = OUTPUT_ROOT / "stage03_splits_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(manifest, indent=2, sort_keys=True))
print("Fresh writes from this notebook remain at", OUTPUT_ROOT)
print("03a will extract embeddings and write query-query plus query-gallery WildFusion matrices from these roles.")